# asyncio 的架构体系
我们将 asyncio 的世界拆解为：一个核心、两个关键字、三个支柱。
## 一个核心：事件循环 (Event Loop)
事件循环是 asyncio 的心脏和大脑。
- 本质：它是一个死循环。它不断地检查：“任务 A 好了吗？”“任务 B 好了吗？”
- 职责：它不执行具体的计算，它只负责调度。当任务 A 在等网络返回时，它立刻把 CPU 切换给任务 B。
## 两个关键字：async 与 await
这是你与调度员交流的语言。
- async def：定义一个“协程（Coroutine）”。它不是立即执行的函数，而是一个可以被暂停的任务单。
- await：这是一个特殊的信号。它告诉调度员：“我要等这儿了，你先去忙别的吧，等我这儿有结果了再叫我。”
## 三个支柱：协程、任务、未来
- Coroutine (协程)：你写的函数。它是静态的，不跑起来就是一堆字节码。
- Task (任务)：协程的“运行实例”。通过 create_task 把协程扔进循环，它才真正开始跑。
- Future (未来对象)：一个占位符。代表“现在还没结果，但未来一定会有”。Ray 的 ObjectRef 本质上就是一种跨进程的 Future。

## 写出第一个协程

In [ ]:
import asyncio

async def say_hello():
    print("Hello...")
    await asyncio.sleep(1) # 模拟 IO 等待，不能用 time.sleep
    print("...World!")

if __name__ == "__main__":
    # asyncio.run 是唯一的入口，它会开启事件循环
    asyncio.run(say_hello())

## 实现并发

In [ ]:
async def task_a():
    await asyncio.sleep(2)
    return "A 完成"

async def task_b():
    await asyncio.sleep(1)
    return "B 完成"

async def main():
    # 两个任务同时开始计时
    # 总耗时是 2 秒，而不是 3 秒
    results = await asyncio.gather(task_a(), task_b())
    print(results) # ['A 完成', 'B 完成']
    
if __name__ == "__main__":
    asyc = asyncio.run(main())

## 后台执行

In [ ]:
async def bg_job():
    await asyncio.sleep(5)
    print("后台任务偷偷跑完了")

async def main():
    # 丢进后台，不阻塞后续代码
    job = asyncio.create_task(bg_job())
    
    print("我先去干别的活了")
    await asyncio.sleep(1)
    print("干完了")
    # 如果你想确保后台任务最后一定完成：
    await job

## 异常和超时处理

In [ ]:
async def slow_api():
    await asyncio.sleep(10)
    return "OK"

async def main():
    try:
        # 只等 2 秒，不等了
        result = await asyncio.wait_for(slow_api(), timeout=2.0)
    except asyncio.TimeoutError:
        print("太慢了，取消任务！")